# WHM — window functions (Fig. 3)

Analytic sheet/filament/halo window functions of section 3.4 of the paper; fully self-contained.


In [ ]:
import os
FIG_DIR = "figures"
os.makedirs(FIG_DIR, exist_ok=True)

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
import scipy.special
from scipy.special import sici
import scipy.integrate as integrate


In [ ]:
nhalos = 64**3
nparticlesperhalo = 512
boxsize = 100
nz = nhalos / boxsize**3 


In [ ]:
def W_TH(k, R):
     return 3./(k*R)**3*(np.sin(k*R) - k*R*np.cos(k*R))

def W_G(k, R):
     return np.exp(-(k*R)**2)

def W_disk(k, a, b, beta):
    j1 = scipy.special.j1(k*a*np.sin(beta))
    return 2*j1/(k*a*np.sin(beta))*np.sin(k*b*np.cos(beta)/2.)/(k*b*np.cos(beta)/2.)

def W_disk_she(k, a, beta):
    j1 = scipy.special.j1(k*a*np.sin(beta))
    return 2*j1/(k*a*np.sin(beta))

def W_disk_fil(k, b, beta):
    return np.sin(k*b*np.cos(beta)/2.)/(k*b*np.cos(beta)/2.)

def W_disk_she_av_ana(k, a):
    Hgr = scipy.special.hyp0f1(2, -(a*k)**2)
    return (2 - 2*Hgr)/(a*k)**2

def W_disk_fil_av_ana(k, b):
    si, ci = scipy.special.sici(k*b)
    return 2*(-1 + np.cos(b*k) + b*k*si)/(b*k)**2

def W_disk_av(k, a, b):
    result_arr = np.ones(len(k))
    for i in range(0,len(k)):
        result_arr[i], err = integrate.quad(lambda x: W_disk(k[i], a, b, x)**2*np.sin(x), 0, np.pi/2.)
    return result_arr

def W_disk_she_av(k, a):
    result_arr = np.ones(len(k))
    for i in range(0,len(k)):
        result_arr[i], err = integrate.quad(lambda x: W_disk_she(k[i], a, x)**2*np.sin(x), 0, np.pi/2.)
    return result_arr

def W_disk_fil_av(k, b):
    result_arr = np.ones(len(k))
    for i in range(0,len(k)):
        result_arr[i], err = integrate.quad(lambda x: W_disk_fil(k[i], b, x)**2*np.sin(x), 0, np.pi/2.)
    return result_arr

def window_halo_nfw(k, R, c, Deltav):
    pi = np.pi

    rv = R / Deltav**(1.0/3.0)
    temp1 = c * R
    temp2 = k * rv / c
    temp3 = k * (R + rv / c)
    temp4 = c * np.sin(k * R)
    temp5 = c * R + rv

    output = -(((temp1 + rv) * (k * np.cos(temp2) * sici(temp3)[1] - temp4 / temp5 - 
               k * (np.cos(temp2) * sici(temp2)[1] + np.sin(temp2) * sici(temp2)[0]) + 
               k * np.sin(temp2) * sici(temp3)[0])) / 
               (k * (temp1 + (temp1 + rv) * np.log(rv / (temp1 + rv)))))

    return output


In [ ]:
import matplotlib

# esthetic definitions for the plots
font = {'size'   : 21, 'family':'STIXGeneral'}
axislabelfontsize='HUGE'
matplotlib.rc('font', **font)
matplotlib.rc('text', usetex=True)
plt.rcParams['axes.linewidth'] = 2
plt.rcParams['legend.fontsize']='small'
plt.rcParams["figure.figsize"] = [8.0,6.0]

kvec = np.logspace(np.log10(0.07),np.log10(10),num=500)

Rval = 10 
haloradius = Rval
sn = 0.0

Deltav_fid = 200

sfac = 2./3.*(5./3.)**0.5
ffac = np.sqrt(5./6.)

# --- Figure Generation ---
fig = plt.figure(figsize=(8, 8))
gs = gridspec.GridSpec(2, 1, height_ratios=[3, 1], hspace=0.05)

ax0 = plt.subplot(gs[0])
ax1 = plt.subplot(gs[1], sharex=ax0)

ax0.loglog(kvec,np.ones(len(kvec)),c='grey',lw=2,ls=':',label=r'0D Point, ($\delta_D^3$)')
ax0.loglog(kvec,(window_halo_nfw(kvec, Rval/Deltav_fid**(1/3), 4, Deltav_fid))**2+sn,c='C3',lw=2,ls='-',label=r'NFW Halo $(c=4)$')

ax0.loglog(kvec,(W_disk_fil_av_ana(kvec, 2*Rval))+sn,c='C2',lw=2,ls=':',label='1D Line ($\delta_D^2$)')
ax0.loglog(kvec,(W_disk_fil_av_ana(kvec, 2*Rval)*window_halo_nfw(kvec, Rval/Deltav_fid**(1/3), 4, Deltav_fid)**2)+sn,c='C2',lw=2,ls='--',label=r'Fil. (thin web)')
ax0.loglog(kvec,(W_disk_fil_av_ana(kvec, 2*Rval)*np.abs(W_TH(kvec, ffac*Rval/Deltav_fid**(1/3)))**2)+sn,c='C2',lw=2,ls='-.',label=r'Fil. (approx)')
ax0.loglog(kvec,(W_disk_av(kvec, np.sqrt(2./3.)*Rval/Deltav_fid**(1/3), 2*Rval))+sn,c='C2',lw=2,ls='-',label=r'Fil. (exact)')

ax0.loglog(kvec,(W_disk_she_av_ana(kvec, Rval))+sn,c='C0',lw=2,ls=':',label='2D Disk ($\delta_D^1$)')
ax0.loglog(kvec,(W_disk_she_av_ana(kvec, Rval)*window_halo_nfw(kvec, Rval/Deltav_fid**(1/3), 4, Deltav_fid)**2)+sn,c='C0',lw=2,ls='--',label=r'She. (thin web)')
ax0.loglog(kvec,(W_disk_she_av_ana(kvec, Rval)*np.abs(W_TH(kvec, sfac*Rval/Deltav_fid**(1/3)))**2)+sn,c='C0',lw=2,ls='-.',label=r'She. (approx)')
ax0.loglog(kvec,(W_disk_av(kvec, Rval, 4./3.*Rval/Deltav_fid**(1/3)))+sn,c='C0',lw=2,ls='-',label=r'She. (exact)')

ax0.loglog(kvec,(W_TH(kvec, Rval))**2+sn,c='k',lw=2,ls='-',label=r'3D Sphere $R$')

ax1.axhline(0.0,c='grey',alpha=0.5)

ax1.semilogx(kvec,(W_disk_she_av_ana(kvec, Rval)*np.abs(W_TH(kvec, sfac*Rval/Deltav_fid**(1/3)))**2)-(W_disk_av(kvec, Rval, 4./3.*Rval/Deltav_fid**(1/3))),c='C0',lw=2,ls='-.')
ax1.semilogx(kvec,(W_disk_fil_av_ana(kvec, 2*Rval)*np.abs(W_TH(kvec, ffac*Rval/Deltav_fid**(1/3)))**2)-(W_disk_av(kvec, np.sqrt(2./3.)*Rval/Deltav_fid**(1/3), 2*Rval)),c='C2',lw=2,ls='-.')
ax1.semilogx(kvec,(W_disk_she_av_ana(kvec, Rval)*window_halo_nfw(kvec, Rval/Deltav_fid**(1/3), 4, Deltav_fid)**2)-(W_disk_av(kvec, Rval, 4./3.*Rval/Deltav_fid**(1/3))),c='C0',lw=2,ls='--')
ax1.semilogx(kvec,(W_disk_fil_av_ana(kvec, 2*Rval)*window_halo_nfw(kvec, Rval/Deltav_fid**(1/3), 4, Deltav_fid)**2)-(W_disk_av(kvec, np.sqrt(2./3.)*Rval/Deltav_fid**(1/3), 2*Rval)),c='C2',lw=2,ls='--')
ax1.semilogx(kvec,(W_disk_she_av_ana(kvec, Rval))-(W_disk_av(kvec, Rval, 4./3.*Rval/Deltav_fid**(1/3))),c='C0',lw=2,ls=':')
ax1.semilogx(kvec,(W_disk_fil_av_ana(kvec, 2*Rval))-(W_disk_av(kvec, np.sqrt(2./3.)*Rval/Deltav_fid**(1/3), 2*Rval)),c='C2',lw=2,ls=':')

print(W_disk_she_av_ana(kvec, Rval)[0]-(W_disk_av(kvec, Rval, 4./3.*Rval/Deltav_fid**(1/3))[0]))

ax0.tick_params(labelbottom=False)

sn = 1./(nhalos*nparticlesperhalo / boxsize**3)
ax0.set_xscale('log')
ax0.set_yscale('symlog', linthresh=sn/50)

ax0.legend(loc='lower left',fontsize=12)
ax0.set_xlim([0.07,10.0])
ax1.set_ylim([-0.0125,0.0125])
ax1.set_xlabel(r'$k\,\,\,\,[h\mathrm{Mpc}^{-1}]$')

ax0.set_ylabel(r'$W^2(k, R=10\,h^{-1}\mathrm{Mpc}, \Delta_\mathrm{v}=200)$')
ax1.set_ylabel(r'$W_\mathrm{cyl}^2-W_\mathrm{exact}^2$')
plt.show()
fig.savefig(os.path.join(FIG_DIR, 'WHMcode_wk_v3.png'), bbox_inches='tight')
